In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [31]:
# Load CSV
try:
    df = pd.read_csv("../../Datasets/csv/UAVUnderNormalCyberattacks.csv", low_memory=False)
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/UAVUnderNormalCyberattacks.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [32]:
# Constants
RANDOM_STATE = 1082
TEST_SIZE = 0.8
BATCH_SIZE = 64
EPOCHS = 400
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [33]:
# Data Preprocessing

# Drop columns
drop_cols = ['timestamp_c', 'class']

# Print unique labels before encoding
print("Unique labels before encoding:", df['class'].unique())

# Remove rows where 'class' is NaN or is the string 'class'
df = df[df['class'].notna()]
df = df[df['class'] != 'class']

# Print unique labels again after cleaning
print("Unique labels after cleaning:", df['class'].unique())

X = df.drop(columns=drop_cols).apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode labels because of string values
le = LabelEncoder()
y = le.fit_transform(df['class'])

print("Unique classes after encoding:", np.unique(y))
num_classes = len(np.unique(y))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

Unique labels before encoding: ['benign' nan 'class' 'DoS attack' 'Replay']
Unique labels after cleaning: ['benign' 'DoS attack' 'Replay']
Unique classes after encoding: [0 1 2]


In [34]:
# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [35]:
# Define a simple MLP model for classification

class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Sigmoid(),
            nn.Linear(128, 64),
            nn.Sigmoid(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)


model = MLP(X_train.shape[1], num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [36]:
# Training loop

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {epoch_loss:.4f}")

Epoch 1/400, Loss: 1.0978
Epoch 2/400, Loss: 1.0900
Epoch 3/400, Loss: 1.0859
Epoch 4/400, Loss: 1.0824
Epoch 5/400, Loss: 1.0796
Epoch 6/400, Loss: 1.0770
Epoch 7/400, Loss: 1.0745
Epoch 8/400, Loss: 1.0722
Epoch 9/400, Loss: 1.0699
Epoch 10/400, Loss: 1.0676
Epoch 11/400, Loss: 1.0653
Epoch 12/400, Loss: 1.0629
Epoch 13/400, Loss: 1.0606
Epoch 14/400, Loss: 1.0582
Epoch 15/400, Loss: 1.0558
Epoch 16/400, Loss: 1.0534
Epoch 17/400, Loss: 1.0510
Epoch 18/400, Loss: 1.0485
Epoch 19/400, Loss: 1.0459
Epoch 20/400, Loss: 1.0433
Epoch 21/400, Loss: 1.0408
Epoch 22/400, Loss: 1.0381
Epoch 23/400, Loss: 1.0354
Epoch 24/400, Loss: 1.0327
Epoch 25/400, Loss: 1.0299
Epoch 26/400, Loss: 1.0272
Epoch 27/400, Loss: 1.0245
Epoch 28/400, Loss: 1.0218
Epoch 29/400, Loss: 1.0191
Epoch 30/400, Loss: 1.0164
Epoch 31/400, Loss: 1.0136
Epoch 32/400, Loss: 1.0109
Epoch 33/400, Loss: 1.0081
Epoch 34/400, Loss: 1.0052
Epoch 35/400, Loss: 1.0025
Epoch 36/400, Loss: 0.9997
Epoch 37/400, Loss: 0.9968
Epoch 38/4

In [37]:
# Evaluation

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(DEVICE)
        outputs = model(batch_X)
        _, preds = torch.max(outputs, 1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch_y.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)
accuracy = accuracy_score(y_true, y_pred)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))

Model Accuracy on Test Set: 62.48%
Confusion Matrix:
[[6308 1993 1036]
 [4326 3780 1499]
 [  97  986 6457]]

Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.68      0.63      9337
           1       0.56      0.39      0.46      9605
           2       0.72      0.86      0.78      7540

    accuracy                           0.62     26482
   macro avg       0.62      0.64      0.62     26482
weighted avg       0.61      0.62      0.61     26482

